Bahdanau注意力模型改进了传统序列到序列模型中固定上下文变量的限制。在每个解码时间步$t'$，它使用一个动态的上下文变量$\mathbf{c}_{t'}$，该变量是注意力机制的输出：

$$\mathbf{c}_{t'} = \sum_{t=1}^T \alpha(\mathbf{s}_{t' - 1}, \mathbf{h}_t) \mathbf{h}_t$$

In [4]:
import torch
from torch import nn

定义注意力解码器

定义Bahdanau注意力，实现循环神经网络编码器-解码器。

为了更方便地显示学习的注意力权重， 以下AttentionDecoder类定义了带有注意力机制解码器的基本接口。

In [5]:
class AttentionDecoder(torch.nn.Module):
    """带有注意力机制解码器的基本接口"""
    def __init__(self, **kwargs):
        super(AttentionDecoder, self).__init__(**kwargs)

    @property
    def attention_weights(self):
        raise NotImplementedError

接下来的Seq2SeqAttentionDecoder类中实现带有Bahdanau注意力的循环神经网络解码器。

In [6]:
class AdditiveAttention(nn.Module):
    """加性注意力"""
    def __init__(self, key_size, query_size, num_hiddens, dropout, **kwargs):
        super(AdditiveAttention, self).__init__(**kwargs)
        self.W_k = nn.Linear(key_size, num_hiddens, bias=False)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=False)
        self.w_v = nn.Linear(num_hiddens, 1, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values, valid_lens):
        queries, keys = self.W_q(queries), self.W_k(keys)
        # 在维度扩展后，queries的形状：(batch_size，查询的个数，1，num_hiddens)
        # keys的形状：(batch_size，1，“键-值”对的个数，num_hiddens)
        # 使用广播方式进行求和
        features = queries.unsqueeze(2) + keys.unsqueeze(1)
        features = torch.tanh(features)
        # self.w_v仅有一个输出，因此从形状中移除最后那个维度。
        # scores的形状：(batch_size，查询的个数，“键-值”对的个数)
        scores = self.w_v(features).squeeze(-1)
        self.attention_weights = masked_softmax(scores, valid_lens)
        # values的形状：(batch_size，“键-值”对的个数，值的维度)
        # output的形状：(batch_size，查询的个数，值的维度)
        return torch.bmm(self.dropout(self.attention_weights), values)

def masked_softmax(X, valid_lens):
    """通过在最后一个轴上屏蔽元素来执行softmax操作"""
    # X: 3D张量，valid_lens: 1D或2D张量
    if valid_lens is None:
        return nn.functional.softmax(X, dim=-1)
    else:
        shape = X.shape
        if valid_lens.dim() == 1:
            valid_lens = torch.repeat_interleave(valid_lens,
                                                shape[1]).reshape(-1, shape[1])
        # 最后一轴上被掩蔽的元素使用一个非常大的负值替换，从而其softmax输出为0
        mask = torch.zeros_like(X, dtype=torch.bool)
        for i in range(shape[0]):
            for j in range(shape[1]):
                mask[i, j, valid_lens[i, j]:] = True
        X_masked = X.masked_fill(mask, -1e6)
        return nn.functional.softmax(X_masked, dim=-1)


In [10]:
class Seq2SeqAttentionDecoder(AttentionDecoder):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers,
                 dropout=0, **kwargs):
        super(Seq2SeqAttentionDecoder, self).__init__(**kwargs)
        # Instantiate AdditiveAttention from the current context
        self.attention = AdditiveAttention(
            num_hiddens, num_hiddens, num_hiddens, dropout)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(
            embed_size + num_hiddens, num_hiddens, num_layers,
            dropout=dropout)
        self.dense = nn.Linear(num_hiddens, vocab_size)

    def init_state(self, enc_outputs, enc_valid_lens, *args):
        # outputs的形状为(batch_size，num_steps，num_hiddens).
        # hidden_state的形状为(num_layers，batch_size，num_hiddens)
        outputs, hidden_state = enc_outputs
        # 保持enc_outputs为(batch_size, num_steps, num_hiddens)
        return (outputs, hidden_state, enc_valid_lens)

    def forward(self, X, state):
        # enc_outputs的形状为(batch_size,num_steps,num_hiddens).
        # hidden_state的形状为(num_layers,batch_size,
        # num_hiddens)
        enc_outputs, hidden_state, enc_valid_lens = state
        # 输出X的形状为(num_steps,batch_size,embed_size)
        X = self.embedding(X).permute(1, 0, 2)
        outputs, self._attention_weights = [], []
        for x in X:
            # query的形状为(batch_size,1,num_hiddens)
            query = torch.unsqueeze(hidden_state[-1], dim=1)
            # context的形状为(batch_size,1,num_hiddens)
            context = self.attention(
                query, enc_outputs, enc_outputs, enc_valid_lens)
            # 在特征维度上连结
            x = torch.cat((context, torch.unsqueeze(x, dim=1)), dim=-1)
            # 将x变形为(1,batch_size,embed_size+num_hiddens)
            out, hidden_state = self.rnn(x.permute(1, 0, 2), hidden_state)
            outputs.append(out)
            self._attention_weights.append(self.attention.attention_weights)
        # 全连接层变换后，outputs的形状为
        # (num_steps,batch_size,vocab_size)
        outputs = self.dense(torch.cat(outputs, dim=0))
        return outputs.permute(1, 0, 2), [enc_outputs, hidden_state,
                                          enc_valid_lens]

    @property
    def attention_weights(self):
        return self._attention_weights

使用包含7个时间步的4个序列输入的小批量测试Bahdanau注意力解码器

In [11]:
# 定义测试参数
vocab_size = 10
embed_size = 8
num_hiddens = 16
num_layers = 2
dropout = 0.1
batch_size = 4
num_steps = 7 # encoder input sequence length
num_decoder_steps = 5 # decoder input sequence length

# 创建虚拟编码器输出
enc_outputs_data = torch.randn(batch_size, num_steps, num_hiddens)
hidden_state_data = torch.randn(num_layers, batch_size, num_hiddens)
enc_valid_lens_data = torch.tensor([num_steps, num_steps - 1, num_steps - 2, num_steps], dtype=torch.long)
enc_outputs = (enc_outputs_data, hidden_state_data)

# 创建虚拟解码器输入
X_decoder = torch.randint(0, vocab_size, (batch_size, num_decoder_steps), dtype=torch.long)

# 实例化解码器
decoder = Seq2SeqAttentionDecoder(vocab_size, embed_size, num_hiddens, num_layers, dropout)

# 初始化解码器状态
state = decoder.init_state(enc_outputs, enc_valid_lens_data)

# 执行前向传播
output, state = decoder(X_decoder, state)

# 打印输出形状和注意力权重形状
print("Decoder output shape:", output.shape)
print("Attention weights len:", len(decoder.attention_weights))
if len(decoder.attention_weights) > 0:
    print("Attention weights shape for first step:", decoder.attention_weights[0].shape)

Decoder output shape: torch.Size([4, 5, 10])
Attention weights len: 5
Attention weights shape for first step: torch.Size([4, 1, 7])


训练



指定超参数，实例化一个带有Bahdanau注意力的编码器和解码器， 并对这个模型进行机器翻译训练

In [ ]:
embed_size, num_hiddens, num_layers, dropout = 32, 32, 2, 0.1
batch_size, num_steps = 64, 10
lr, num_epochs, device = 0.005, 250, d2l.try_gpu()

train_iter, src_vocab, tgt_vocab = d2l.load_data_nmt(batch_size, num_steps)
encoder = d2l.Seq2SeqEncoder(
    len(src_vocab), embed_size, num_hiddens, num_layers, dropout)
decoder = Seq2SeqAttentionDecoder(
    len(tgt_vocab), embed_size, num_hiddens, num_layers, dropout)
net = d2l.EncoderDecoder(encoder, decoder)
d2l.train_seq2seq(net, train_iter, lr, num_epochs, tgt_vocab, device)

可视化注意力权重，发现每个查询都会在键值对上分配不同的权重，这说明 在每个解码步中，输入序列的不同部分被选择性地聚集在注意力池中。